# Data Processing and Merging Notebook


    This notebook contains the steps taken to process and merge employee data from multiple Excel sheets.
    The process involves cleaning, merging, and analyzing data from different sources to achieve the final output.
    Each step is explained in detail with accompanying code.
    

## Step 1: Load and Clean Region Data

In [ ]:

import pandas as pd

# Load the 'Central USA' sheet from the original dataset
central_usa_df = pd.read_excel('Case 3 - Demand Managenet.xlsx', sheet_name='Central USA')

# Drop the first two rows which are headers and unrelated data
central_usa_cleaned = central_usa_df.drop([0, 1])

# Rename the first column as 'Employee Code' and set it as the identifier
central_usa_cleaned.rename(columns={'Unnamed: 0': 'Employee Code'}, inplace=True)

# Reset index to easily navigate rows
central_usa_cleaned.reset_index(drop=True, inplace=True)


## Step 2: Iterate Through Data and Calculate Metrics

In [ ]:

# Initialize lists to store the calculated metrics
employee_ids = []
total_contracted_hours = []
total_allocated_hours = []
total_remaining_capacity = []
average_utilization_rate = []

# Iterate through the DataFrame to extract and calculate the values
i = 0
while i < len(central_usa_cleaned):
    if "EMP_" in central_usa_cleaned.loc[i, "Employee Code"]:
        emp_id = central_usa_cleaned.loc[i, "Employee Code"]
        contracted_hours = central_usa_cleaned.iloc[i + 1, 1:].astype(float).sum()
        allocated_hours = central_usa_cleaned.iloc[i + 2, 1:].astype(float).sum()
        remaining_capacity = contracted_hours - allocated_hours
        utilization_rate = (allocated_hours / contracted_hours) * 100 if contracted_hours > 0 else 0

        # Append the results
        employee_ids.append(emp_id)
        total_contracted_hours.append(contracted_hours)
        total_allocated_hours.append(allocated_hours)
        total_remaining_capacity.append(remaining_capacity)
        average_utilization_rate.append(utilization_rate)

        # Skip the next rows (including tasks)
        i += 6
    else:
        i += 1

# Create the final DataFrame with the correct structure
final_region_df = pd.DataFrame({
    'Employee Code': employee_ids,
    'Total Contracted Hours': total_contracted_hours,
    'Total Allocated Hours': total_allocated_hours,
    'Total Remaining Capacity': total_remaining_capacity,
    'Average Utilization Rate': average_utilization_rate
})


## Step 3: Merge with Employee Dept Data

In [ ]:

# Load the Employee Dept sheet data
employee_dept = pd.read_excel('Case 3 - Demand Managenet.xlsx', sheet_name='Employee Dept')

# Rename and clean up the Employee Dept data
employee_dept.columns = ['Ignore', 'Employee Code', 'Department', 'Grade', 'Level']
employee_dept_cleaned = employee_dept.drop(columns=['Ignore'])

# Merge this data with the cleaned region data
merged_with_dept = pd.merge(final_region_df, employee_dept_cleaned, on='Employee Code', how='left')


## Step 4: Merge with Employee HR Cost Data

In [ ]:

# Load the Employee HR Cost sheet data
employee_hr_cost = pd.read_excel('Case 3 - Demand Managenet.xlsx', sheet_name='Employee HR Cost')

# Rename and clean up the Employee HR Cost data
employee_hr_cost.columns = ['Ignore', 'Employee Code', 'Cost']
employee_hr_cost_cleaned = employee_hr_cost.drop(columns=['Ignore'])

# Merge this data with the existing merged data
final_merged_data = pd.merge(merged_with_dept, employee_hr_cost_cleaned, on='Employee Code', how='left')


## Step 5: Save the Final Data

In [ ]:

# Save the final merged data to a new Excel file
final_merged_data.to_excel('Final_Combined_Regions_With_Blue_Sheet_Info.xlsx', index=False)
